# Gold: dim_customers

## Import Helper Functions

In [1]:
from src.config_loader import load_config
from src.spark_sql_magic import sql
from src.gold.helper import get_changed_customer_ids
from src.gold.dims.customers import build_dim_customers_scd2, build_incremental_dim_customers, validate_scd2_customers
from src.watermark import  get_last_commit_ts, get_effective_watermark
from src.writers import overwrite_table, replace_by_key

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Load Configs

In [2]:
cfg = load_config()

JOB_NAMES = cfg["spark_jobs"]["jobs"]
CATALOG = cfg["general"]["catalog"]
SILVER_NAMESPACE = cfg["general"]["namespaces"]["silver"]
GOLD_NAMESPACE = cfg["general"]["namespaces"]["gold"]

cfg_cust= cfg["gold"]["dim_customers_scd2"]
SOURCE_TABLE = cfg_cust["source_table"]
TARGET_TABLE = cfg_cust["target_table"]
GEO_TABLE = cfg_cust["geo_table"]
BUFFER_HOURS = cfg_cust["buffer_hours"]
KEY_COLUMNS = cfg_cust["key_columns"]

## Import Libraries and Start Session

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["gold_customers"])
        .getOrCreate()
)

## Run Pipeline

In [4]:
def run_dim_customers_pipeline(spark):
    print("[START] dim_customers pipeline")
    
    last_commit_ts = get_last_commit_ts(spark, TARGET_TABLE)
    print(f"[INFO] last_commit_ts = {last_commit_ts}")
    
    if last_commit_ts is None:
        print("[INFO] first run → full rebuild")
        df = build_dim_customers_scd2(spark, SOURCE_TABLE, GEO_TABLE)
        validate_scd2_customers(df)
        overwrite_table(df, TARGET_TABLE)
        print("[END] full rebuild complete")
        return

    effective_ts = get_effective_watermark(last_commit_ts, BUFFER_HOURS)
    changed_customer_ids = get_changed_customer_ids(spark, effective_ts)

    if changed_customer_ids.isEmpty():
        print("[INFO] no changes detected → skip")
        return
    print("[INFO] changes detected → incremental run")

    df = build_incremental_dim_customers(spark, SOURCE_TABLE, GEO_TABLE, changed_customer_ids)
    validate_scd2_customers(df)
    replace_by_key(spark, df, TARGET_TABLE, KEY_COLUMNS)
    print("[END] incremental update complete")    

In [5]:
# if __name__ == "__main__":
#     from pyspark.sql import SparkSession

#     spark = SparkSession.builder.getOrCreate()
run_dim_customers_pipeline(spark)

[START] dim_customers pipeline
[INFO] last_commit_ts = None
[INFO] first run → full rebuild


[END] full rebuild complete


## Sanity Check

In [6]:
%%sql
SHOW TABLES IN polaris.gold;

+---------+------------------+-----------+
|namespace|tableName         |isTemporary|
+---------+------------------+-----------+
|gold     |dim_customers_scd2|false      |
+---------+------------------+-----------+



In [7]:
%%sql
SELECT * FROM polaris.gold.dim_customers_scd2 
LIMIT 10

+--------------------------------+--------------------------------+------------------------+-------------------+--------------+-------------------+-------------------+-----------------------+-----------------------+-----------------------+------------+----------+----------------------------------------------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city      |customer_state|geolocation_lat    |geolocation_lng    |cdc_ts                 |spark_ingest_ts        |effective_from         |effective_to|is_current|customer_sk                                                     |
+--------------------------------+--------------------------------+------------------------+-------------------+--------------+-------------------+-------------------+-----------------------+-----------------------+-----------------------+------------+----------+----------------------------------------------------------------+
|00050bf6e01e

In [8]:
spark.catalog.clearCache()  # clears all cached tables
spark.stop() 